# 化学实验流程验证 Playground

本notebook用于测试从plan_logs中读取的化学实验流程，验证工作站顺序是否满足约束条件。

In [ ]:
import yaml
from sample import Sample
import re

In [ ]:
# 简化的工作站名称到约束条件的映射
WORKSTATION_CONSTRAINTS = {
    '物料站': {
        'constraint': lambda sample: True,  # 无约束
        'description': '获取50ml离心管'
    },
    '1ml液体进样站': {
        'constraint': lambda sample: (
            sample.data['container']['container_name'] == "rack" and
            sample.data['container']['subcontainer']['subcontainer_name'] in ["50ml_centrifuge_tube", "ptfe_reaction_bottle", "vial"] and
            sample.data['container']['subcontainer']['subcontainer_number'] is not None and
            1 <= sample.data['container']['subcontainer']['subcontainer_number'] <= 10 and
            sample.data['container']['subcontainer']['subcontainer_volume'] <= 30 and
            sample.data['container']['subcontainer']['subcontainer_phase'] in ['liquid', 'suspension', 'solid']
        ),
        'description': '添加1ml液体'
    },
    '磁力搅拌工作站': {
        'constraint': lambda sample: (
            sample.data['container']['container_name'] == "rack" and
            sample.data['container']['subcontainer']['subcontainer_name'] in ["50ml_centrifuge_tube", "ptfe_reaction_bottle", "vial"] and
            sample.data['container']['subcontainer']['subcontainer_number'] is not None and
            1 <= sample.data['container']['subcontainer']['subcontainer_number'] <= 10 and
            sample.data['container']['subcontainer']['subcontainer_phase'] in ['liquid', 'suspension'] and
            sample.data['container']['subcontainer']['subcontainer_volume'] > 0 and
            sample.data['container']['subcontainer']['subcontainer_volume'] <= 30
        ),
        'description': '搅拌混合'
    },
    '超声清洗工作站': {
        'constraint': lambda sample: (
            sample.data['container']['container_name'] == "rack" and
            sample.data['container']['subcontainer']['subcontainer_name'] in ["50ml_centrifuge_tube", "ptfe_reaction_bottle", "vial"] and
            sample.data['container']['subcontainer']['subcontainer_phase'] in ['liquid', 'suspension'] and
            sample.data['container']['subcontainer']['subcontainer_volume'] > 0
        ),
        'description': '超声清洗'
    },
    '离心纯化工作站': {
        'constraint': lambda sample: (
            sample.data['container']['container_name'] == "rack" and
            sample.data['container']['subcontainer']['subcontainer_name'] in ["50ml_centrifuge_tube", "ptfe_reaction_bottle", "vial"] and
            sample.data['container']['subcontainer']['subcontainer_phase'] == 'suspension' and
            sample.data['container']['subcontainer']['subcontainer_volume'] > 0
        ),
        'description': '离心分离'
    },
    '烘干机': {
        'constraint': lambda sample: (
            sample.data['container']['container_name'] == "rack" and
            sample.data['container']['subcontainer']['subcontainer_name'] in ["50ml_centrifuge_tube", "ptfe_reaction_bottle", "vial"] and
            sample.data['container']['subcontainer']['subcontainer_phase'] in ['solid', 'suspension']
        ),
        'description': '加热干燥'
    },
    '高通量红外光谱工作站': {
        'constraint': lambda sample: (
            sample.data['container']['container_name'] == "rack" and
            sample.data['container']['subcontainer']['subcontainer_name'] in ["50ml_centrifuge_tube", "ptfe_reaction_bottle", "vial"] and
            sample.data['container']['subcontainer']['subcontainer_phase'] in ['solid', 'liquid']
        ),
        'description': '红外光谱分析'
    },
    '高通量XRD工作站': {
        'constraint': lambda sample: (
            sample.data['container']['container_name'] == "rack" and
            sample.data['container']['subcontainer']['subcontainer_name'] in ["50ml_centrifuge_tube", "ptfe_reaction_bottle", "vial"] and
            sample.data['container']['subcontainer']['subcontainer_phase'] == 'solid'
        ),
        'description': 'XRD分析'
    }
}

In [ ]:
def parse_plan_log(log_file_path):
    """
    解析plan log文件，提取工作站步骤
    """
    steps = []
    with open(log_file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 提取每个步骤的方法名
    pattern = r"'method': '([^']+)'"
    matches = re.findall(pattern, content)

    for match in matches:
        steps.append(match)

    return steps

In [ ]:
# 解析plan_1.log
log_file = "./plan_logs_withRAG/1/plan_1.log"
steps = parse_plan_log(log_file)
print("从日志文件解析的工作站步骤:")
for i, step in enumerate(steps, 1):
    print(f"  {i}. {step}")

In [ ]:
# 创建初始样品
sample = Sample()
print("初始样品状态:")
print(f"容器类型: {sample.data['container']['container_name']}")
print(f"子容器: {sample.data['container']['subcontainer']['subcontainer_name']}")
print(f"子容器体积: {sample.data['container']['subcontainer']['subcontainer_volume']} ml")
print(f"相态: {sample.data['container']['subcontainer']['subcontainer_phase']}")

In [ ]:
# 模拟执行第一步：物料站
print("\n=== 步骤 1: 物料站 ===")
step_name = steps[0]
constraint_info = WORKSTATION_CONSTRAINTS[step_name]
print(f"工作站描述: {constraint_info['description']}")

if constraint_info['constraint'](sample):
    print("✅ 约束条件通过")
    # 模拟操作：获取50ml离心管
    sample.data['container']['container_name'] = "rack"
    sample.data['container']['subcontainer']['subcontainer_name'] = '50ml_centrifuge_tube'
    sample.data['container']['subcontainer']['subcontainer_number'] = 1
    sample.data['container']['subcontainer']['subcontainer_volume'] = 0
    sample.data['container']['subcontainer']['subcontainer_phase'] = 'solid'
    print("✅ 步骤 1 执行成功")
else:
    print("❌ 约束条件不满足")

print("\n步骤 1 执行后的样品状态:")
print(f"容器类型: {sample.data['container']['container_name']}")
print(f"子容器: {sample.data['container']['subcontainer']['subcontainer_name']}")
print(f"子容器体积: {sample.data['container']['subcontainer']['subcontainer_volume']} ml")
print(f"相态: {sample.data['container']['subcontainer']['subcontainer_phase']}")

In [ ]:
# 模拟执行第二步：1ml液体进样站
print("\n=== 步骤 2: 1ml液体进样站 ===")
step_name = steps[1]
constraint_info = WORKSTATION_CONSTRAINTS[step_name]
print(f"工作站描述: {constraint_info['description']}")

if constraint_info['constraint'](sample):
    print("✅ 约束条件通过")
    # 模拟操作：添加1ml液体
    if sample.data['container']['subcontainer']['subcontainer_phase'] == 'solid':
        sample.data['container']['subcontainer']['subcontainer_phase'] = 'suspension'
    sample.data['container']['subcontainer']['subcontainer_volume'] += 1
    print("✅ 步骤 2 执行成功")
else:
    print("❌ 约束条件不满足")

print("\n步骤 2 执行后的样品状态:")
print(f"容器类型: {sample.data['container']['container_name']}")
print(f"子容器: {sample.data['container']['subcontainer']['subcontainer_name']}")
print(f"子容器体积: {sample.data['container']['subcontainer']['subcontainer_volume']} ml")
print(f"相态: {sample.data['container']['subcontainer']['subcontainer_phase']}")

In [ ]:
# 验证整个流程
def validate_flow(steps, sample):
    """
    验证整个工作流程的可行性
    """
    print("\n=== 化学实验流程验证开始 ===")
    print(f"总步骤数: {len(steps)}")

    for i, step_name in enumerate(steps, 1):
        print(f"\n{'='*50})")
        print(f"正在执行步骤 {i}: {step_name}")

        # 获取约束条件
        if step_name not in WORKSTATION_CONSTRAINTS:
            print(f"❌ 错误: 未知的工作站 '{step_name}'")
            return False

        constraint_info = WORKSTATION_CONSTRAINTS[step_name]
        print(f"工作站描述: {constraint_info['description']}")

        # 检查约束条件
        print(f"检查约束条件...")
        try:
            if constraint_info['constraint'](sample):
                print(f"✅ 约束条件通过")

                # 根据步骤模拟状态变化
                if step_name == '物料站':
                    sample.data['container']['container_name'] = "rack"
                    sample.data['container']['subcontainer']['subcontainer_name'] = '50ml_centrifuge_tube'
                    sample.data['container']['subcontainer']['subcontainer_number'] = 1
                    sample.data['container']['subcontainer']['subcontainer_volume'] = 0
                    sample.data['container']['subcontainer']['subcontainer_phase'] = 'solid'
                elif step_name == '1ml液体进样站':
                    if sample.data['container']['subcontainer']['subcontainer_phase'] == 'solid':
                        sample.data['container']['subcontainer']['subcontainer_phase'] = 'suspension'
                    sample.data['container']['subcontainer']['subcontainer_volume'] += 1
                elif step_name == '烘干机':
                    sample.data['temperature'] = 60
                    sample.data['container']['subcontainer']['subcontainer_phase'] = 'solid'
                
                print(f"✅ 步骤 {i} 执行成功")
            else:
                print(f"❌ 步骤 {i} 约束条件不满足")
                return False
        except Exception as e:
            print(f"❌ 步骤 {i} 检查约束时出错: {str(e)}")
            return False

    print(f"\n{'='*50})")
    print("✅ 整个流程验证成功！")
    return True

# 创建新的样品实例并验证整个流程
new_sample = Sample()
success = validate_flow(steps, new_sample)

if success:
    print("\n🎉 化学实验流程验证完成 - 成功!")
else:
    print("\n❌ 化学实验流程验证失败!")

## 使用说明

1. **解析plan log**: 从`plan_logs_withRAG/X/plan_Y.log`文件中读取工作站步骤
2. **验证每个步骤**: 检查每个工作站的约束条件是否满足
3. **模拟状态变化**: 根据工作站功能更新样品状态
4. **评估整个流程**: 判断整个实验流程是否可行

## 约束条件说明

- **物料站**: 无约束，获取50ml离心管
- **1ml液体进样站**: 需要离心管架，体积≤30ml，相态为液体/悬浊液/固体
- **磁力搅拌工作站**: 需要液体或悬浊液，体积>0且≤30ml
- **超声清洗工作站**: 需要液体或悬浊液，体积>0
- **离心纯化工作站**: 需要悬浊液，体积>0
- **烘干机**: 需要固体或悬浊液
- **高通量红外光谱工作站**: 需要固体或液体
- **高通量XRD工作站**: 需要固体